<a href="https://colab.research.google.com/github/StanleyLiangYork/Journal-Club-26-Applying-Multimodal-LLM-for-Medical-Research/blob/main/tensorflow_gpt2_gpt3_basic_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-2 and GPT-3 Basic Architecture in TensorFlow/Keras

This notebook mirrors the PyTorch notebook, but implements the same educational task with TensorFlow and Keras.

The notebook demonstrates:

- Reading `/content/textdata.txt`
- Building a character-level tokenizer
- Encoding text into token IDs
- Implementing causal multi-head self-attention
- Building GPT-style decoder-only Transformer models
- Creating toy GPT-2-like and GPT-3-like models
- Predicting the next token
- Generating text of a chosen length
- Training the toy models briefly
- Saving and loading the trained Toy GPT-3-like model for inference

The real GPT-2 and GPT-3 models are much larger than the toy models used here. The purpose is to teach the architecture and workflow, not to reproduce full-scale training.

## 1. Imports and Reproducibility

In [1]:
from dataclasses import asdict, dataclass
from pathlib import Path
import json

import tensorflow as tf

tf.random.set_seed(7)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Load the Text Dataset

The requested input path is a common Google Colab path. Upload the file to this location before running the cell.

In [2]:
TEXT_PATH = Path("/content/textdata.txt")

def load_text_dataset(path=TEXT_PATH):
    """Read a UTF-8 text file from disk."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. In Colab, upload textdata.txt to this path first."
        )
    return path.read_text(encoding="utf-8", errors="replace")


text_data = load_text_dataset(TEXT_PATH)
print(f"Loaded {len(text_data):,} characters")
print(text_data[:500])

Loaded 161 characters
Welcome to Natural Language Processing  
It is one of the most exciting research areas as of today  
We will see how Python can be used to work with text files. 


## 3. Character-Level Tokenizer

Real GPT models use subword tokenization. A character-level tokenizer keeps the code easy to inspect for a journal club demonstration.

In [3]:
class CharTokenizer:
    """Minimal character tokenizer for educational language modeling."""

    def __init__(self, text=None, chars=None):
        if chars is None:
            chars = sorted(set(text))
        self.chars = list(chars)
        self.stoi = {ch: i for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}

    @property
    def vocab_size(self):
        return len(self.chars)

    def encode(self, text):
        """Convert text into a list of integer token IDs."""
        return [self.stoi[ch] for ch in text if ch in self.stoi]

    def decode(self, ids):
        """Convert token IDs back into text."""
        return "".join(self.itos[int(i)] for i in ids)


tokenizer = CharTokenizer(text=text_data)
data_ids = tf.constant(tokenizer.encode(text_data), dtype=tf.int32)

print("Vocabulary size:", tokenizer.vocab_size)
print("Encoded dataset shape:", data_ids.shape)
print("First 100 decoded characters:")
print(tokenizer.decode(data_ids[:100].numpy().tolist()))

Vocabulary size: 29
Encoded dataset shape: (161,)
First 100 decoded characters:
Welcome to Natural Language Processing  
It is one of the most exciting research areas as of today  


## 4. GPT Configuration

In [4]:
@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0

    @property
    def head_dim(self):
        assert self.n_embd % self.n_head == 0
        return self.n_embd // self.n_head


# Real model metadata for discussion only. Do not instantiate these here.
GPT2_SMALL_REAL = GPTConfig(vocab_size=50_257, block_size=1024, n_layer=12, n_head=12, n_embd=768)
GPT3_175B_REAL = GPTConfig(vocab_size=50_257, block_size=2048, n_layer=96, n_head=96, n_embd=12_288)

# Runnable toy models using the dataset vocabulary.
GPT2_TOY = GPTConfig(vocab_size=tokenizer.vocab_size, block_size=64, n_layer=2, n_head=4, n_embd=32)
GPT3_TOY = GPTConfig(vocab_size=tokenizer.vocab_size, block_size=64, n_layer=4, n_head=4, n_embd=64)

print("GPT-2 small metadata:", GPT2_SMALL_REAL)
print("GPT-3 175B metadata:", GPT3_175B_REAL)
print("Toy GPT-2-like config:", GPT2_TOY)
print("Toy GPT-3-like config:", GPT3_TOY)

GPT-2 small metadata: GPTConfig(vocab_size=50257, block_size=1024, n_layer=12, n_head=12, n_embd=768, dropout=0.0)
GPT-3 175B metadata: GPTConfig(vocab_size=50257, block_size=2048, n_layer=96, n_head=96, n_embd=12288, dropout=0.0)
Toy GPT-2-like config: GPTConfig(vocab_size=29, block_size=64, n_layer=2, n_head=4, n_embd=32, dropout=0.0)
Toy GPT-3-like config: GPTConfig(vocab_size=29, block_size=64, n_layer=4, n_head=4, n_embd=64, dropout=0.0)


## 5. Causal Multi-Head Self-Attention

Self-attention lets each token compare itself with previous tokens. The causal mask blocks access to future tokens, which is essential for left-to-right text generation.

In [5]:
class CausalSelfAttention(tf.keras.layers.Layer):
    """Masked multi-head self-attention for GPT-style models."""

    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.head_dim = config.head_dim
        self.n_embd = config.n_embd

        # One dense layer computes queries, keys, and values together.
        self.qkv = tf.keras.layers.Dense(3 * config.n_embd)
        self.proj = tf.keras.layers.Dense(config.n_embd)
        self.dropout = tf.keras.layers.Dropout(config.dropout)

    def split_heads(self, x):
        """Convert (B, T, C) into (B, n_head, T, head_dim)."""
        batch_size = tf.shape(x)[0]
        seq_len = tf.shape(x)[1]
        x = tf.reshape(x, [batch_size, seq_len, self.n_head, self.head_dim])
        return tf.transpose(x, [0, 2, 1, 3])

    def merge_heads(self, x):
        """Convert (B, n_head, T, head_dim) back into (B, T, C)."""
        batch_size = tf.shape(x)[0]
        seq_len = tf.shape(x)[2]
        x = tf.transpose(x, [0, 2, 1, 3])
        return tf.reshape(x, [batch_size, seq_len, self.n_embd])

    def call(self, x, training=False):
        seq_len = tf.shape(x)[1]

        qkv = self.qkv(x)
        q, k, v = tf.split(qkv, 3, axis=-1)
        q = self.split_heads(q)
        k = self.split_heads(k)
        v = self.split_heads(v)

        scores = tf.matmul(q, k, transpose_b=True)
        scores = scores / tf.math.sqrt(tf.cast(self.head_dim, tf.float32))

        # Lower-triangular mask has 1 for allowed positions and 0 for future positions.
        mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        scores = tf.where(mask[None, None, :, :] == 1, scores, tf.fill(tf.shape(scores), -1e9))

        weights = tf.nn.softmax(scores, axis=-1)
        weights = self.dropout(weights, training=training)
        y = tf.matmul(weights, v)
        y = self.merge_heads(y)
        return self.proj(y)

## 6. Transformer Decoder Block

In [6]:
class FeedForward(tf.keras.layers.Layer):
    """Position-wise MLP used after attention."""

    def __init__(self, config):
        super().__init__()
        self.fc = tf.keras.layers.Dense(4 * config.n_embd, activation=tf.keras.activations.gelu)
        self.proj = tf.keras.layers.Dense(config.n_embd)
        self.dropout = tf.keras.layers.Dropout(config.dropout)

    def call(self, x, training=False):
        x = self.fc(x)
        x = self.proj(x)
        return self.dropout(x, training=training)


class TransformerBlock(tf.keras.layers.Layer):
    """One GPT decoder block with pre-layer normalization."""

    def __init__(self, config):
        super().__init__()
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.attn = CausalSelfAttention(config)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.mlp = FeedForward(config)

    def call(self, x, training=False):
        x = x + self.attn(self.ln1(x), training=training)
        x = x + self.mlp(self.ln2(x), training=training)
        return x

## 7. Full GPT Language Model

In [7]:
class GPTLanguageModel(tf.keras.Model):
    """Small GPT-style character language model."""

    def __init__(self, config):
        super().__init__()
        self.config_obj = config
        self.token_embedding = tf.keras.layers.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = tf.keras.layers.Embedding(config.block_size, config.n_embd)
        self.blocks = [TransformerBlock(config) for _ in range(config.n_layer)]
        self.ln_f = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.lm_head = tf.keras.layers.Dense(config.vocab_size)

    def call(self, idx, training=False):
        seq_len = tf.shape(idx)[1]
        positions = tf.range(seq_len)

        token_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(positions)[None, :, :]
        x = token_emb + pos_emb

        for block in self.blocks:
            x = block(x, training=training)

        x = self.ln_f(x)
        return self.lm_head(x)

    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Autoregressively sample new token IDs."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config_obj.block_size:]
            logits = self(idx_cond, training=False)
            logits = logits[:, -1, :] / max(temperature, 1e-8)

            if top_k is not None:
                values, _ = tf.math.top_k(logits, k=top_k)
                cutoff = values[:, -1:]
                logits = tf.where(logits < cutoff, tf.fill(tf.shape(logits), -1e9), logits)

            next_id = tf.random.categorical(logits, num_samples=1, dtype=tf.int32)
            idx = tf.concat([idx, next_id], axis=1)

        return idx

## 8. Instantiate Toy GPT-2-like and GPT-3-like Models

In [8]:
def build_model(config):
    """Create and build a model by running one dummy forward pass."""
    model = GPTLanguageModel(config)
    dummy = tf.zeros((1, min(4, config.block_size)), dtype=tf.int32)
    _ = model(dummy, training=False)
    return model


def count_parameters(model):
    """Count trainable scalar parameters."""
    return int(sum(tf.size(var).numpy() for var in model.trainable_variables))


gpt2_model = build_model(GPT2_TOY)
gpt3_model = build_model(GPT3_TOY)

print(f"Toy GPT-2-like parameters: {count_parameters(gpt2_model):,}")
print(f"Toy GPT-3-like parameters: {count_parameters(gpt3_model):,}")

Toy GPT-2-like parameters: 29,405
Toy GPT-3-like parameters: 207,901


## 9. Encode a Prompt and Predict the Next Token

In [9]:
def show_next_token_candidates(model, prompt, tokenizer, top_k=5):
    """Print top next-token candidates for a prompt."""
    ids = tokenizer.encode(prompt)
    if len(ids) == 0:
        raise ValueError("Prompt has no characters found in the tokenizer vocabulary.")

    idx = tf.constant([ids[-model.config_obj.block_size:]], dtype=tf.int32)
    logits = model(idx, training=False)
    probs = tf.nn.softmax(logits[0, -1], axis=-1)
    values, indices = tf.math.top_k(probs, k=top_k)

    print("Prompt:")
    print(prompt)
    print("\nEncoded prompt IDs:")
    print(idx[0].numpy().tolist())
    print("\nTop next-token candidates:")
    for prob, token_id in zip(values.numpy().tolist(), indices.numpy().tolist()):
        print(f"id={token_id:>3} token={tokenizer.decode([token_id])!r} probability={prob:.4f}")


prompt = text_data[:80]

print("Toy GPT-2-like next-token prediction")
show_next_token_candidates(gpt2_model, prompt, tokenizer)

print("\nToy GPT-3-like next-token prediction")
show_next_token_candidates(gpt3_model, prompt, tokenizer)

Toy GPT-2-like next-token prediction
Prompt:
Welcome to Natural Language Processing  
It is one of the most exciting research

Encoded prompt IDs:
[8, 18, 1, 4, 8, 20, 14, 25, 8, 14, 12, 1, 6, 22, 21, 10, 12, 23, 23, 16, 20, 14, 1, 1, 0, 3, 24, 1, 16, 23, 1, 21, 20, 12, 1, 21, 13, 1, 24, 15, 12, 1, 19, 21, 23, 24, 1, 12, 27, 10, 16, 24, 16, 20, 14, 1, 22, 12, 23, 12, 8, 22, 10, 15]

Top next-token candidates:
id= 24 token='t' probability=0.1333
id= 16 token='i' probability=0.1267
id=  6 token='P' probability=0.0966
id=  3 token='I' probability=0.0762
id= 27 token='x' probability=0.0741

Toy GPT-3-like next-token prediction
Prompt:
Welcome to Natural Language Processing  
It is one of the most exciting research

Encoded prompt IDs:
[8, 18, 1, 4, 8, 20, 14, 25, 8, 14, 12, 1, 6, 22, 21, 10, 12, 23, 23, 16, 20, 14, 1, 1, 0, 3, 24, 1, 16, 23, 1, 21, 20, 12, 1, 21, 13, 1, 24, 15, 12, 1, 19, 21, 23, 24, 1, 12, 27, 10, 16, 24, 16, 20, 14, 1, 22, 12, 23, 12, 8, 22, 10, 15]

Top next-token candi

## 10. Generate a Given Length of Text

In [10]:
def generate_text(model, prompt, tokenizer, max_new_tokens=200, temperature=1.0, top_k=10):
    """Encode a prompt, generate new token IDs, and decode back to text."""
    ids = tokenizer.encode(prompt)
    idx = tf.constant([ids], dtype=tf.int32)
    generated = model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(generated[0].numpy().tolist())


GENERATE_LENGTH = 200

print("Toy GPT-2-like generated text:")
print(generate_text(gpt2_model, prompt, tokenizer, max_new_tokens=GENERATE_LENGTH))

print("\nToy GPT-3-like generated text:")
print(generate_text(gpt3_model, prompt, tokenizer, max_new_tokens=GENERATE_LENGTH))

Toy GPT-2-like generated text:
Welcome to Natural Language Processing  
It is one of the most exciting researchNtbitWkPtkxttiePxIPWuitPfIcuxxtttttIictctPchctttcWecPccelkhhgccWhPPkPteielotkkkgiPhgeeWgtgcWtWhhgctWPtetthcWtreWWtWtrWhWPhtWtt
ttWWNWWthttWeWWtgchtWgiclheWhgihWhhigchthNetWhlNNtmihtellhNmlhrcgNroWiccr

Toy GPT-3-like generated text:
Welcome to Natural Language Processing  
It is one of the most exciting researchWwLeWWekgLLmWllk WWwwWNWkWLPPeewg kePLwLeeeWLeewmiWwgsWoa
ffwgwfwume
rn
sgnwttnt
tgattto
o
r.ot
m
oett

geiLLkwwmwtsgadgLswmwlwa
wattmtatmt

mkmwwdmotti
tfofgoggowtikwiwe
gegoe
iwoewgweiiLwwuowogLwioo


## 11. Mini-Batches for Next-Token Training

In [11]:
def get_batch(data_ids, batch_size=16, block_size=64):
    """Create mini-batches for next-token prediction."""
    if int(tf.shape(data_ids)[0]) <= block_size + 1:
        raise ValueError("Dataset is too short for the requested block_size.")

    max_start = int(tf.shape(data_ids)[0]) - block_size - 1
    starts = tf.random.uniform((batch_size,), minval=0, maxval=max_start, dtype=tf.int32)
    x = tf.stack([data_ids[i : i + block_size] for i in starts])
    y = tf.stack([data_ids[i + 1 : i + block_size + 1] for i in starts])
    return x, y


batch_x, batch_y = get_batch(data_ids, batch_size=2, block_size=GPT2_TOY.block_size)
print("x shape:", batch_x.shape)
print("y shape:", batch_y.shape)
print("Example input text:")
print(tokenizer.decode(batch_x[0].numpy().tolist()))
print("Example target text, shifted by one token:")
print(tokenizer.decode(batch_y[0].numpy().tolist()))

x shape: (2, 64)
y shape: (2, 64)
Example input text:
tural Language Processing  
It is one of the most exciting resea
Example target text, shifted by one token:
ural Language Processing  
It is one of the most exciting resear


## 12. Training Helpers

In [12]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

def train_toy_model(model, config, data_ids, train_steps=100, batch_size=16, learning_rate=3e-4):
    """Tiny custom training loop for next-token prediction."""
    optimizer = tf.keras.optimizers.AdamW(learning_rate=learning_rate)

    for step in range(train_steps):
        x, y = get_batch(data_ids, batch_size=batch_size, block_size=config.block_size)

        with tf.GradientTape() as tape:
            logits = model(x, training=True)
            loss = loss_fn(y, logits)

        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        if step % 25 == 0 or step == train_steps - 1:
            print(f"step {step:>3} | loss {loss.numpy():.4f}")

    return model

## 13. Train Toy GPT-2-like Model

In [13]:
TRAIN_STEPS = 100
LEARNING_RATE = 3e-4

gpt2_model = train_toy_model(
    gpt2_model,
    GPT2_TOY,
    data_ids,
    train_steps=TRAIN_STEPS,
    batch_size=16,
    learning_rate=LEARNING_RATE,
)

print("\nToy GPT-2-like generated text after tiny training loop:")
print(generate_text(gpt2_model, prompt, tokenizer, max_new_tokens=GENERATE_LENGTH))

step   0 | loss 4.2537
step  25 | loss 2.8854
step  50 | loss 2.6014
step  75 | loss 2.1818
step  99 | loss 1.8661

Toy GPT-2-like generated text after tiny training loop:
Welcome to Natural Language Processing  
It is one of the most exciting researchingis 
an oLas otithiark  wNad w of ecsexy  
Wilg uanisestily 
Wh
stinIbeseseus odase wit e  owiy serorexceas ust  u.onacg  
n f  tf tingt eyn usgyf thon e ong  m teiconif orll ccstorhehititheseseadas


## 14. Train Toy GPT-3-like Model

In [14]:
gpt3_model = train_toy_model(
    gpt3_model,
    GPT3_TOY,
    data_ids,
    train_steps=TRAIN_STEPS,
    batch_size=16,
    learning_rate=LEARNING_RATE,
)

print("\nToy GPT-3-like generated text after tiny training loop:")
print(generate_text(gpt3_model, prompt, tokenizer, max_new_tokens=GENERATE_LENGTH))

step   0 | loss 4.0381
step  25 | loss 2.7141
step  50 | loss 2.1342
step  75 | loss 1.3951
step  99 | loss 0.9391

Toy GPT-3-like generated text after tiny training loop:
Welcome to Natural Language Processing  
It is one of the most exciting research earch eas of as mof os 
We e tof 
ay  towin Pexs
Wex Pyth calll to twithon witoseedan Pyt Pyth bexcit PytPyse cith it sexcixexcare urk  WexWeWe reIeexn willlserdan Pythof caredan s Py  wis  sex ust t


## 15. Save the Trained Toy GPT-3 Model

Keras weights save only the learned parameters. We also save the model configuration and tokenizer vocabulary as JSON so the same architecture can be reconstructed later.

In [15]:
CHECKPOINT_DIR = Path("toy_gpt3_tf_checkpoint")
CHECKPOINT_DIR.mkdir(exist_ok=True)

WEIGHTS_PATH = CHECKPOINT_DIR / "toy_gpt3.weights.h5"
METADATA_PATH = CHECKPOINT_DIR / "metadata.json"

# Save learned TensorFlow/Keras weights.
gpt3_model.save_weights(str(WEIGHTS_PATH))

# Save architecture and tokenizer information.
metadata = {
    "config": asdict(GPT3_TOY),
    "chars": tokenizer.chars,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved weights to: {WEIGHTS_PATH.resolve()}")
print(f"Saved metadata to: {METADATA_PATH.resolve()}")

Saved weights to: /content/toy_gpt3_tf_checkpoint/toy_gpt3.weights.h5
Saved metadata to: /content/toy_gpt3_tf_checkpoint/metadata.json


## 16. Load the Saved Toy GPT-3 Model for Inference

In [16]:
loaded_metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
loaded_config = GPTConfig(**loaded_metadata["config"])
loaded_tokenizer = CharTokenizer(chars=loaded_metadata["chars"])

loaded_gpt3_model = build_model(loaded_config)
loaded_gpt3_model.load_weights(str(WEIGHTS_PATH))

print("Loaded config:", loaded_config)
print(f"Loaded tokenizer vocabulary size: {loaded_tokenizer.vocab_size}")
print(f"Loaded model parameters: {count_parameters(loaded_gpt3_model):,}")

Loaded config: GPTConfig(vocab_size=29, block_size=64, n_layer=4, n_head=4, n_embd=64, dropout=0.0)
Loaded tokenizer vocabulary size: 29
Loaded model parameters: 207,901


## 17. Inference with the Loaded Toy GPT-3 Model

In [17]:
print("Loaded Toy GPT-3-like next-token prediction:")
show_next_token_candidates(loaded_gpt3_model, prompt, loaded_tokenizer)

print("\nLoaded Toy GPT-3-like generated text:")
print(generate_text(
    loaded_gpt3_model,
    prompt,
    loaded_tokenizer,
    max_new_tokens=GENERATE_LENGTH,
    temperature=1.0,
    top_k=10,
))

Loaded Toy GPT-3-like next-token prediction:
Prompt:
Welcome to Natural Language Processing  
It is one of the most exciting research

Encoded prompt IDs:
[8, 18, 1, 4, 8, 20, 14, 25, 8, 14, 12, 1, 6, 22, 21, 10, 12, 23, 23, 16, 20, 14, 1, 1, 0, 3, 24, 1, 16, 23, 1, 21, 20, 12, 1, 21, 13, 1, 24, 15, 12, 1, 19, 21, 23, 24, 1, 12, 27, 10, 16, 24, 16, 20, 14, 1, 22, 12, 23, 12, 8, 22, 10, 15]

Top next-token candidates:
id=  1 token=' ' probability=0.6548
id= 12 token='e' probability=0.2396
id= 21 token='o' probability=0.0693
id= 14 token='g' probability=0.0069
id= 17 token='k' probability=0.0050

Loaded Toy GPT-3-like generated text:
Welcome to Natural Language Processing  
It is one of the most exciting research t as mof wof theaos s tof tho th witit Py cl 
Wexcaytowitof cil cin ytouseexcaredWexcare w oreeedarkexrk cdarexchkw uf 
as as Pywiise th chof Pk thof Pytiil c
We titil se case w rebe Pyoree bearexciu


## 18. Journal Club Discussion Prompts

1. Which TensorFlow/Keras layers correspond to the PyTorch modules?
2. Why does the causal mask make next-token prediction valid?
3. Why are GPT-2 and GPT-3 better described as different scales of the same core architecture?
4. What would change if this notebook used subword tokenization instead of character tokenization?
5. What are the limitations of text-only GPT models for multimodal medical research?